# Data Cleaning Utilities
# 
## What we do here:
##   - Rename columns to consistent names
##   - Fix data types (customer_id → integer, create_date → date)
##  - Standardize messy categorical values 
##     (gender: M/male/1 → "Male", F/female/2 → "Female")
##    (marital_status: S/single → "Single", M/married → "Married")

## Why:
##   Make raw data clean, consistent and ready for analysis / modeling
##  Prevent errors caused by wrong types or different spellings

In [0]:
# import libraries and functions

import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType, IntegerType, DateType


# Remove leading and trailing spaces from all string columns
def trim_string_columns(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, trim(col(field.name)))
    return df


# Rename columns using provided mapping dictionary
def rename_columns(df, rename_map):
    for old_name, new_name in rename_map.items():
        if old_name in df.columns:
            df = df.withColumnRenamed(old_name, new_name)
    return df


# Cast selected columns to correct data types
def cast_columns(df):
    return (
        df
        .withColumn("customer_id", col("customer_id").cast(IntegerType()))
        .withColumn("create_date", col("create_date").cast(DateType()))
    )


# Standardize categorical values (gender and marital_status)
def standardize_categorical(df):
    return (
        df
        .withColumn(
            "marital_status",
            F.when(F.upper(F.col("marital_status")).isin("S", "SINGLE"), "Single")
             .when(F.upper(F.col("marital_status")).isin("M", "MARRIED"), "Married")
             .otherwise(None)
        )
        .withColumn(
            "gender",
            F.when(F.upper(F.col("gender")).isin("M", "MALE", "1"), "Male")
             .when(F.upper(F.col("gender")).isin("F", "FEMALE", "2"), "Female")
             .otherwise(None)
        )
    )